In [1]:
import pandas as pd
import numpy as np

# Read data after outcome processing
df_project = pd.read_csv(
    "../data/data_processed/df_outcomes_processed.csv",
    low_memory=False
)

print("Original shape:", df_project.shape)

Original shape: (14512, 62)


In [2]:
## 1. Protective behaviour predictors
# They are dummy encoded during final data processing


## 2. Demographic predictors
# Age
df_project["age_num"] = pd.to_numeric(
    df_project["age"],
    errors="coerce"
)

# Gender 
# It is dummy encoded during final data processing


## 3. Household predictors
# Household size
# "8 or more" -> 8
def convert_household_size(x):
    if x in [str(i) for i in range(1, 8)]:
        return int(x)
    elif x == "8 or more":
        return 8
    else:
        return np.nan

df_project["household_size_num"] = (
    df_project["household_size"]
    .apply(convert_household_size)
)

# Household children
# 0 = no children
# 1 = one child
# 2 = two or more children
children_map = {
    "0": "0",
    "1": "1",
    "2": "2_or_more",
    "3": "2_or_more",
    "4": "2_or_more",
    "5 or more": "2_or_more"
}

df_project["household_children_group"] = (
    df_project["household_children"]
    .map(children_map)
)


## 4. Employment predictors

# Convert employment status variables to binary values
# Yes = 1, No = 0

employment_map = {
    "Yes": 1,
    "No": 0
}

employment_vars = [
    f"employment_status_{i}" for i in range(1, 8)
]

for col in employment_vars:
    df_project[col] = df_project[col].map(employment_map)


## 5. Contact predictors

contact_vars = [
    "i1_health",
    "i2_health",
    "i7a_health"
]

for col in contact_vars:
    df_project[col] = pd.to_numeric(
        df_project[col],
        errors="coerce"
    )


## 6. Testing and COVID experience
# Nominal multi-category variables.

personal_test_status_map = {
    "Yes, and I tested positive": "positive",
    "Yes, and I tested negative": "negative",
    "Yes, and I have not received my results from the test yet": "pending",
    "No, I have not": "not_tested",
    "Not sure": "not_sure"
}

household_test_status_map = {
    "Yes, and they tested positive": "positive",
    "Yes, and they tested negative": "negative",
    "Yes, and they have not received their results from the test yet": "pending",
    "No, they have not": "not_tested",
    "Not sure": "not_sure"
}

df_project["personal_test_status"] = (
    df_project["i3_health"]
    .map(personal_test_status_map)
)

df_project["household_test_status"] = (
    df_project["i4_health"]
    .map(household_test_status_map)
)

In [3]:
## 7. COVID-like symptom predictors

i5_cols = [
    col for col in df_project.columns
    if col.startswith("i5_health_")
]

symptom_cols = [
    f"i5_health_{i}" for i in range(1, 6)
]

# Identify whether any specific symptom was reported
has_symptom = df_project[symptom_cols].eq("Yes").any(axis=1)

# Identify contradictory responses:
# one or more symptoms reported AND "None of these" selected
inconsistent_i5 = (
    has_symptom
    & (df_project["i5_health_99"] == "Yes")
)

print("Inconsistent symptom responses:", inconsistent_i5.sum())

df_project["covid_symptom_status"] = None

# At least one symptom reported
df_project.loc[
    has_symptom,
    "covid_symptom_status"
] = "Yes"

# None of these selected
df_project.loc[
    df_project["i5_health_99"] == "Yes",
    "covid_symptom_status"
] = "No"

# Contradictory responses are treated as missing
df_project.loc[
    inconsistent_i5,
    "covid_symptom_status"
] = None

print(df_project["covid_symptom_status"].value_counts(dropna=False))

df_project = df_project.drop(columns=i5_cols)

Inconsistent symptom responses: 0
covid_symptom_status
No      13588
Yes       697
None      227
Name: count, dtype: int64


In [4]:
## 8. Comorbidity predictors

d1_cols = [
    col for col in df_project.columns
    if col.startswith("d1_")
]

d1_condition_cols = [
    f"d1_health_{i}" for i in range(1, 14)
]

# Whether the respondent reported at least one specific condition
has_comorbidity = df_project[d1_condition_cols].eq("Yes").any(axis=1)

# Whether "None of these" or "Prefer not to say" was selected
selected_none = df_project["d1_health_99"] == "Yes"
selected_pns = df_project["d1_health_98"] == "Yes"

# Identify contradictory responses
inconsistent_d1 = (
    (has_comorbidity & selected_none)
    | (has_comorbidity & selected_pns)
    | (selected_none & selected_pns)
)

print("Inconsistent comorbidity responses:", inconsistent_d1.sum())

df_project["comorbidity_status"] = None

# At least one specific comorbidity reported
df_project.loc[
    has_comorbidity,
    "comorbidity_status"
] = "Yes"

# None of these selected
df_project.loc[
    selected_none,
    "comorbidity_status"
] = "No"

# Prefer not to say selected
df_project.loc[
    selected_pns,
    "comorbidity_status"
] = "Prefer_not_to_say"

# Contradictory responses are treated as missing
df_project.loc[
    inconsistent_d1,
    "comorbidity_status"
] = None

print(df_project["comorbidity_status"].value_counts(dropna=False))

df_project = df_project.drop(columns=d1_cols)

Inconsistent comorbidity responses: 0
comorbidity_status
No                   12124
Yes                   1978
Prefer_not_to_say      394
None                    16
Name: count, dtype: int64


In [5]:
## 9. Time predictor

df_project["survey_wave"] = (
    df_project["qweek"]
    .str.extract(r"(\d+)", expand=False)
    .astype("Int64")
)

In [6]:
## 10. Save predictor-processed dataset

df_project.to_csv(
    "../data/data_processed/df_predictors_processed.csv",
    index=False
)